In [ ]:
# type: ignore

import os
import requests 
import json
import ollama
from typing import List
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display,clear_output

In [2]:
# Constants

MODEL = "llama3.2"

In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [9]:
def get_links(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
        ], format = "json"  #Define format as json!
    )
    result = response['message']['content']

    return json.loads(result)


In [10]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

anthropic = Website("https://anthropic.com")
anthropic.links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/claude',
 'https://www.anthropic.com/max',
 'https://www.anthropic.com/team',
 'https://www.anthropic.com/enterprise',
 'https://www.anthropic.com/pricing',
 'https://claude.ai/download',
 'https://claude.ai/',
 'https://www.anthropic.com/news/claude-character',
 'https://www.anthropic.com/api',
 'https://docs.anthropic.com/',
 'https://www.anthropic.com/pricing#api',
 'https://console.anthropic.com/',
 'https://docs.anthropic.com/en/docs/welcome',
 'https://www.anthropic.com/solutions/agents',
 'https://www.anthropic.com/solutions/coding',
 'https://www.anthropic.com/solutions/customer-support',
 'https://www.anthropic.com/solutions/education',
 'https://www.anthropic.com/solutions/financial-services',
 'https://www.anthropic.com/customers',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-index',
 'https://www.anthropic.com/claude/opus',
 'https://www.anthropic.com/claude/sonne

In [11]:
get_links("https://anthropic.com")

{'links': [{'type': 'Company page', 'url': 'https://www.anthropic.com/'},
  {'type': "About founder Claude's website", 'url': 'https://claude.ai/'},
  {'type': "Max's website (founder)", 'url': 'https://www.anthropic.com/max'},
  {'type': 'Team page', 'url': 'https://www.anthropic.com/team'},
  {'type': 'Enterprise page', 'url': 'https://www.anthropic.com/enterprise'},
  {'type': 'Pricing page', 'url': 'https://www.anthropic.com/pricing'},
  {'type': "News page with Claude's character",
   'url': 'https://www.anthropic.com/news/claude-character'},
  {'type': 'API documentation', 'url': 'https://docs.anthropic.com/'},
  {'type': 'Console for Anthropic AI',
   'url': 'https://console.anthropic.com/'},
  {'type': 'News page with visible extended thinking',
   'url': 'https://www.anthropic.com/news/visible-extended-thinking'},
  {'type': 'Transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'Responsible scaling policy',
   'url': 'https://www.anthropic.com/re

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [13]:
print(get_all_details("https://anthropic.com"))

Found links: {'links': [{'type': 'About page', 'url': 'https://www.anthropic.com/'}, {'type': 'Company page', 'url': 'https://www.anthropic.com/company'}, {'type': 'Careers/Jobs page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'News page', 'url': 'https://www.anthropic.com/news'}, {'type': 'Research page', 'url': 'https://www.anthropic.com/research'}, {'type': 'Blog', 'url': 'https://www.anthropic.com/blog'}, {'type': 'GitHub repository', 'url': 'https://github.com/anthropicai'}, {'type': 'Docs/Documentation page', 'url': 'https://docs.anthropic.com/'}]}
Landing page:
Webpage Title:
Home \ Anthropic
Webpage Contents:
Skip to main content
Skip to footer
Claude
Chat with Claude
Overview
Max plan
Team plan
Enterprise plan
Explore pricing
Download apps
Claude log in
News
Claude’s character
API
Build with Claude
API overview
Developer docs
Explore pricing
Console log in
News
Learn how to build with Claude
Solutions
Collaborate with Claude
AI agents
Coding
Customer support
Educat

In [14]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'About page', 'url': 'https://www.anthropic.com/'}, {'type': 'Company page', 'url': 'https://www.anthropic.com/company'}, {'type': 'Careers page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Research', 'url': 'https://www.anthropic.com/research'}, {'type': 'Engineering', 'url': 'https://www.anthropic.com/engineering'}, {'type': 'Solutions/Agents', 'url': 'https://www.anthropic.com/solutions/agents'}, {'type': 'Solutions/Coding', 'url': 'https://www.anthropic.com/solutions/coding'}, {'type': 'Solutions/Customers', 'url': 'https://www.anthropic.com/customers'}, {'type': 'Partners/MCP', 'url': 'https://www.anthropic.com/partners/mcp'}, {'type': 'Partners/Powered by Claude', 'url': 'https://www.anthropic.com/partners/powered-by-claude'}, {'type': 'News', 'url': 'https://www.anthropic.com/news'}, {'type': 'Events', 'url': 'https://www.anthropic.com/events'}, {'type': 'Legal/Company Terms', 'url': 'https://www.anthropic.com/legal/commercial-terms'}

'You are looking at a company called: Anthropic\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHome \\ Anthropic\nWebpage Contents:\nSkip to main content\nSkip to footer\nClaude\nChat with Claude\nOverview\nMax plan\nTeam plan\nEnterprise plan\nExplore pricing\nDownload apps\nClaude log in\nNews\nClaude’s character\nAPI\nBuild with Claude\nAPI\xa0overview\nDeveloper docs\nExplore pricing\nConsole log in\nNews\nLearn how to build with Claude\nSolutions\nCollaborate with Claude\nAI\xa0agents\nCoding\nCustomer support\nEducation\nFinancial services\nCase studies\nHear from our customers\nResearch\nResearch\nOverview\nEconomic Index\nClaude model family\nClaude Opus 4\nClaude Sonnet 4\nClaude Haiku 3.5\nResearch\nClaude’s extended thinking\nCommitments\nInitiatives\nTransparency\nResponsible scaling policy\nTrust center\nSecurity and compliance\nAnnouncement\nISO

In [17]:
def create_brochure(company_name, url):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response["message"]["content"]
    display(Markdown(result))


In [18]:
create_brochure("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'About page', 'url': 'https://www.anthropic.com/'}, {'type': 'Company page', 'url': 'https://www.anthropic.com/company'}, {'type': 'Careers/Jobs page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'News page', 'url': 'https://www.anthropic.com/news'}, {'type': 'Research page', 'url': 'https://www.anthropic.com/research'}, {'type': 'Transparency page', 'url': 'https://www.anthropic.com/transparency'}, {'type': 'Responsible scaling policy', 'url': 'https://www.anthropic.com/responsible-scaling-policy'}, {'type': 'Engineering page', 'url': 'https://www.anthropic.com/engineering'}, {'type': 'Learn about Anthropic page', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Customers page', 'url': 'https://www.anthropic.com/customers'}, {'type': 'Solutions page (agents)', 'url': 'https://www.anthropic.com/solutions/agents'}, {'type': 'Solutions page (coding)', 'url': 'https://www.anthropic.com/solutions/coding'}, {'type': 'Solutions page (customer su

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


**Anthropic: Empowering Safe and Responsible AI Development**
===========================================================

**About Us**
------------

At Anthropic, we're dedicated to building AI that serves humanity's long-term well-being. Our mission is to create powerful technologies while prioritizing human benefit and safety.

**Our Values**
--------------

* **Responsible AI Development**: We focus on designing tools with human benefit at their foundation.
* **Intentional Pauses**: We take time to consider the effects of our work, ensuring that our innovations are both bold and responsible.
* **Collaboration**: We partner with experts across industries to drive positive change.

**Our Solutions**
-----------------

### Claude AI

Meet Claude Opus 4, our most intelligent AI model. Built on top of Claude Code, this powerful tool enables developers to create custom experiences and applications that prioritize human benefit.

### Research

* **Anthropic Economic Index**: Track the economic impact of AI on society.
* **Claude models**: Explore our range of AI agents, including Opus 4, Sonnet 4, and Haiku 3.5.

**Join Our Community**
----------------------

If you're passionate about building a better future with AI, join us!

### Careers

We have open roles available in:

* **Engineering**: Help us build the future of safe AI.
* **Sales**: Discuss our solutions with potential customers.
* **Research**: Contribute to our mission-driven research initiatives.

### Partner Directory

Partner with Anthropic to drive positive change in your industry.

**News and Announcements**
-------------------------

Stay up-to-date on our latest news and announcements:

* **Project Vend**: Learn about our collaboration with [project partner].
* **Responsible Scaling Policy**: Discover our commitment to responsible AI development.
* **Anthropic Academy**: Access our learning resources and tutorials.

# Final omprovement

getting a typewriter animation

In [19]:
def create_brochure(company_name, url):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    display_markdown = display(Markdown(""), display_id=True)  # Initialize Markdown display
    response_text = ""

    for chunk in ollama.chat(model=MODEL, messages=messages, stream=True):  # Ensure stream=True (not a string)
        response_text += chunk['message']['content']
        clear_output(wait=True)  # Clear previous output to create a streaming effect
        display_markdown.update(Markdown(response_text))  # Update Markdown dynamically


In [20]:
create_brochure("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'About page', 'url': 'https://www.anthropic.com/'}, {'type': 'Company page', 'url': 'https://www.anthropic.com/company'}, {'type': 'Careers/Jobs page', 'url': 'https://www.anthropic.com/careers'}, {'type': 'Research page', 'url': 'https://www.anthropic.com/research'}, {'type': 'Economic Index page', 'url': 'https://www.anthropic.com/economic-index'}, {'type': 'Engineering page', 'url': 'https://www.anthropic.com/engineering'}, {'type': 'Learn/Documentation page', 'url': 'https://www.anthropic.com/learn'}, {'type': 'Partners page', 'url': 'https://www.anthropic.com/partners/mcp'}, {'type': 'News page', 'url': 'https://www.anthropic.com/news'}, {'type': 'Blog page', 'url': 'https://www.anthropic.com/blog'}, {'type': 'Transparency page', 'url': 'https://www.anthropic.com/transparency'}]}


**Brochure: Anthropic**

[Cover Image: A neural network diagram or a futuristic cityscape]

**Welcome to Anthropic**

At Anthropic, we're on a mission to build AI that serves humanity's long-term well-being. Our company culture is built around collaboration, innovation, and responsible AI development.

**Our Mission**

We believe that designing powerful technologies requires both bold steps forward and intentional pauses to consider the effects. That's why we focus on building tools with human benefit at their foundation, like Claude.

**What We Do**

* Develop cutting-edge AI models, including Claude Opus 4, Sonnet 4, and Haiku 3.5
* Build AI-powered applications and custom experiences using our API platform, Claude Code
* Provide customer support, education, and financial services to help you get the most out of your Claude experience

**Our Commitments**

* Responsible Scaling Policy: We prioritize transparency and safety in our development processes.
* Transparency: Our research and policy work are available for public review.

**Our Team**

We're a team of passionate individuals who share a common goal: to build AI that benefits humanity. If you're interested in joining us, check out our [careers page](#careers).

**Collaborate with Us**

* Try Claude today and experience the power of AI for yourself
* Learn how to build with Claude through our Anthropic Academy
* Partner with us to develop innovative solutions using our API platform

**Stay Informed**

* Follow us on social media to stay up-to-date on our latest research, announcements, and news.
* Subscribe to our newsletter to receive updates on our company culture, products, and events.

[Back Cover: A call-to-action to learn more about Anthropic and how you can collaborate with us]

**Join the Conversation**

What will happen between humans and machines in the future? How can we design technologies that benefit humanity's long-term well-being?

At Anthropic, we're exploring these questions and more. Join the conversation and help us build a better future for AI.

[Disclaimer: This brochure is for informational purposes only and may contain technical inaccuracies or outdated information.]